# **Вариант №18**

3) Найдите одно из решений системы уравнений методом Ньютона с абсолютной погрешностью 0,0001.
$$\begin{cases}
\sin (x-0.6) - y = 1.6\\
3x + \cos y = 0.9
\end{cases}$$

Подключаем библиотеки:

In [2]:
from typing import Callable
from dataclasses import dataclass, asdict
from math import cos, sin

Прописываем классы для удобной работы ввода и вывода данных

In [3]:
@dataclass
class Result:
    x: float | list
    fx: float | list
    count_iter: int

    def __str__(self):
        return f"x = {self.x}, f(x) = {self.fx}, iter = {self.count_iter}"


@dataclass
class Equation:
    func: Callable[..., float]
    func_primes: list[Callable[..., float]]

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class ExampleCasesNewton:
    functions: list[Equation]
    limits: list[tuple[float, float]]
    x0: list[float] = None
    epsilon: float = 10e-5
    max_iteration: float = 10_000

    def to_dict(self) -> dict:
        return asdict(self)

    def __str__(self) -> str:
        return f"Example: limits = {self.limits}, x0 = {self.x0},\
epsilon = {self.epsilon}, max iteration = {self.max_iteration}"

Методы

In [4]:
def jacobian(functions: list[Equation], x: list[float]):
    return [[i(*x) for i in eq.func_primes] for eq in functions]


def func_vector(functions: list[Equation], x: list[float]) -> list[float]:
    return [eq.func(*x) for eq in functions]


def minor(A: list[list[float]], x: int, y: int) -> list[list[float]]:
    B = A[:y] + A[y + 1 :]
    return [mat[:x] + mat[x + 1 :] for mat in B]


def determinate(A: list[list[float]]) -> float:
    y_len = len(A)

    if y_len == 0:
        return 0

    x_len = len(A[0])

    if x_len != y_len:
        raise ValueError("Матрица не является квадратной!")

    if x_len == 1:
        return A[0][0]
    if x_len == 2:
        return A[0][0] * A[1][1] - A[0][1] * A[1][0]
    if x_len == 3:
        return (
            A[0][0] * A[1][1] * A[2][2]
            + A[0][1] * A[1][2] * A[2][0]
            + A[0][2] * A[1][0] * A[2][1]
            - A[0][2] * A[1][1] * A[2][0]
            - A[0][0] * A[1][2] * A[2][1]
            - A[0][1] * A[1][0] * A[2][2]
        )

    S = 0
    for i in range(x_len):
        t = A[0][i] * (-1) ** i * determinate(minor(A, i, 0))
        S += t
    return S


def transposition(A: list[list[float]]) -> list[list[float]]:
    x_len, y_len = len(A[0]), len(A)
    T = [[0] * y_len for _ in range(x_len)]

    for i in range(y_len):
        for j in range(x_len):
            T[j][i] = A[i][j]

    return T


def matrix_minors(A: list[list[float]]) -> list[list[float]]:
    x_len, y_len = len(A[0]), len(A)
    M = [[0] * x_len for _ in range(y_len)]

    for i in range(y_len):
        for j in range(x_len):
            M[j][i] = determinate(minor(A, i, j)) * (-1) ** (i + j)

    return M


def inverse_matrix_minors(A: list[list[float]]):
    if len(A) == 1:
        return [[1 / A[0][0]]]

    # 1. Находим определитель матрицы
    det_a = determinate(A)
    if det_a == 0:
        raise ValueError(
            "Обратная матрица для матрицы с нулевым определителем не существует!"
        )

    # 2.Находим матрицу алгебраических дополнений
    B = matrix_minors(A)

    # 3. Транспонируем матрицу алгебраических дополнений

    B_T = transposition(B)

    # 4. Вывод результата
    result = B_T
    for i in range(len(B_T)):
        for j in range(len(B_T[0])):
            B_T[i][j] *= 1 / det_a

    return result


def inverse_matrix_gauss(A: list[list[float]]) -> list[list[float]]:
    y_len = len(A)

    if not A or y_len == 0:
        raise ValueError("Матрица пустая!")

    x_len = len(A[0])

    if x_len != y_len:
        raise ValueError("Матрица не является квадратной!")

    if y_len == 1:
        if A[0][0] == 0:
            raise ValueError("Обратная матрица для [[0]] не существует!")
        return [[1 / A[0][0]]]

    # 1. Находим определитель матрицы
    det_a = determinate(A)
    if det_a == 0:
        raise ValueError(
            "Обратная матрица для матрицы с нулевым определителем не существует!"
        )

    # Создаем единичную матрицу
    E = [[1 if i == j else 0 for j in range(x_len)] for i in range(y_len)]

    M = [A[i] + E[i] for i in range(y_len)]

    x_len = len(M[0])

    # Приводим матрицу к ступенчатому виду
    for row in range(y_len - 1):
        for sub_row in range(row + 1, y_len):
            coef = M[sub_row][row] / M[row][row]
            for col in range(x_len):
                temp = M[sub_row][col] - coef * M[row][col]
                if abs(temp) < 1e-15:
                    M[sub_row][col] = 0
                else:
                    M[sub_row][col] = temp

    # Приводим диагональ матрицы к единицам
    for row in range(y_len):
        if M[row][row] != 1:
            coef = M[row][row]
            for col in range(x_len):
                M[row][col] /= coef

    # Приводим левую сторону матрицы к единичной
    for row in range(y_len - 1, -1, -1):
        for sub_row in range(row - 1, -1, -1):
            if row < x_len:
                coef = M[sub_row][row]
                if abs(coef) > 1e-15:
                    for col in range(x_len):
                        temp = M[sub_row][col] - coef * M[row][col]
                        if abs(temp) < 1e-15:
                            M[sub_row][col] = 0
                        else:
                            M[sub_row][col] = temp

    return [i[y_len:] for i in M]


def mul_matrix(A: list[list[float]], B: list[list[float]]):
    if len(A[0]) != len(B):
        raise ValueError("Размеры матриц не позволяют произвести умножение")

    m, l, n = len(A[0]), len(A), len(B[0])

    C = [[0.0] * n for _ in range(l)]

    for i in range(l):
        for j in range(n):
            C[i][j] = sum(A[i][r] * B[r][j] for r in range(m))

    return C


def ax_equal_b(
    A: list[list[float]], B: list[list[float]], method: Callable = inverse_matrix_minors
) -> list[list[float]]:
    X = mul_matrix(method(A), B)
    return X

Метод Ньютона для систем:

In [5]:
def method_newton_cases(
    functions: list[Equation],
    limits: list[tuple[float, float]],
    x0: list[float] = None,
    epsilon: float = 10e-5,
    max_iteration: int = 10_000,
) -> Result:
    if x0 is None:
        x0 = [(l[0] + l[1]) / 2 for l in limits]

    x = x0.copy()

    for iteration in range(max_iteration):
        F = func_vector(functions, x)

        if max(map(abs, F)) < epsilon:
            return Result(x=x, fx=F, count_iter=iteration)

        J = jacobian(functions, x)

        try:
            # Исправление: преобразуем F в матрицу-столбец
            F_matrix = [[-f] for f in F]
            delta_matrix = ax_equal_b(J, F_matrix)
            delta = [row[0] for row in delta_matrix]  # Извлекаем из матрицы-столбца

        except ValueError:
            try:
                J_inv = inverse_matrix_gauss(J)
                # Исправление: ручное умножение обратной матрицы на вектор
                delta = []
                for i in range(len(x)):
                    s = 0
                    for j in range(len(x)):
                        s += J_inv[i][j] * (-F[j])
                    delta.append(s)
            except:
                raise ValueError(f"Якобиан вырожден на итерации {iteration}")

        x = [x[i] + delta[i] for i in range(len(x))]

    raise RuntimeError(
        f"Нахождение корня превысило количество допустимых итераций ({max_iteration=})"
    )

Создание объектов задания для тестирования

In [6]:
task = ExampleCasesNewton(
    functions=[
        Equation(
            func=lambda x, y: sin(x - 0.6) - y - 1.6,
            func_primes=[lambda x, y: cos(x - 0.6), lambda x, y: -1],
        ),
        Equation(
            func=lambda x, y: 3 * x + cos(y) - 0.9,
            func_primes=[lambda x, y: 3, lambda x, y: -sin(y)],
        ),
    ],
    limits=[
        (-2, 2),
        (-2, 2),
    ],
)

In [8]:
result = method_newton_cases(
    functions=task.functions,
    limits=task.limits,
    x0=task.x0,
    epsilon=task.epsilon,
    max_iteration=task.max_iteration,
)

print(result)

x = [0.3812763311049309, -1.8170176403743508], f(x) = [3.3762975413687e-05, 8.800783116569288e-05], iter = 3
